# V3-8 — D2: Metadata-only diagnostic and video+metadata fusion

D2 reuses frozen RGB and motion caches; it never reads MP4 files. It is a dataset-specific benchmark: the default model for an arbitrary MP4 remains video-only.

In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from v3_d2_metadata_fusion import Config, build_context, cache_preflight, context_report, run_experiment

config = Config()
RUN_EXPERIMENT = True
print({'data_root': str(config.data_root), 'primary_aggregation': config.primary_aggregation, 'mp4_decoding_required': False})

In [ ]:
# 1) Freeze the data contract and inspect only the allowed metadata fields.
context = build_context(config)
print(context_report(context))
display(context['metadata'].groupby(['split', 'video_label']).size().rename('videos').reset_index())
for field in ('weather', 'light_conditions', 'scene'):
    print(field)
    display(context['metadata'].groupby(['split', field]).size().rename('videos').reset_index())

In [ ]:
# 2) Validate the cached [16,512] RGB and motion features and train-only metadata vocabulary.
preflight = cache_preflight(context)
print(preflight)
assert preflight['mp4_decoding_required'] is False
assert preflight['expected_sequences'] == 3214

In [ ]:
# 3) Run D2-1 metadata-only, D2-2 video-only control, and D2-3 fusion.
if RUN_EXPERIMENT:
    result = run_experiment()
    print(json.dumps(result['summary'], ensure_ascii=False, indent=2))
else:
    print('Set RUN_EXPERIMENT=True to run the CPU-only cached-feature experiment.')

In [ ]:
# 4) Review all three comparisons and the metadata slice/error report.
if RUN_EXPERIMENT:
    print('Metadata-only:')
    print(result['metadata_only']['metrics'])
    display(result['video_only']['aggregation'])
    display(result['fusion']['aggregation'])
    display(result['bias_report'])

Interpretation rule: if metadata-only is strong, treat it as a shortcut/bias warning. D2 must not replace the arbitrary-MP4 video-only model even if its development score is higher.